# MSI Pipeline - Script 02: Create AnnData Objects

This notebook creates unified AnnData objects from MSI parquet files.

## Supported Input Formats

**Single-file format (recommended, default from script01):**
- One parquet file per sample with all channels as columns
- Format: `{input_dir}/{sample_id}.parquet`
- Columns: x, y, <channel_name_1>, <channel_name_2>, ...
- Preserves original channel names exactly

**Per-channel format (legacy):**
- One parquet file per channel
- Format: `{input_dir}/{sample_id}/{channel_name}.parquet`
- Each file has columns: x, y, intensity

The notebook auto-detects which format is present.

## Output
- AnnData objects: `{output_dir}/{sample_id}.h5ad`
  - `.X`: intensity matrix (pixels x channels)
  - `.obs`: x, y, sample_id
  - `.obsm['spatial']`: coordinate array
  - `.var`: channel metadata

In [12]:
import sys
from pathlib import Path
import logging

import numpy as np
import pandas as pd
import anndata as ad

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))

from utils import io as msi_io

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration

In [21]:
# === CONFIGURE THESE PATHS ===

# Base data directory
BASE_DIR = Path(r"T:/Sammy Data/Third set results/")  # Path.home() / "ext_hd_sammy"

# Input: parquet files from script01
PARQUET_DIR = BASE_DIR / "out_msi_peptides" 

# Output: AnnData files
OUTPUT_DIR = BASE_DIR / "out_anndata_peptides"

# Data modality
MODALITY = "peptides"  # Options: glycans, metabolites, peptides

# Join type for merging channels
JOIN_TYPE = "outer"  # 'outer' = union, 'inner' = intersection

# Fill value for missing intensities
FILL_VALUE = 0.0

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {PARQUET_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Modality: {MODALITY}")

Input directory: T:\Sammy Data\Third set results\out_msi_peptides
Output directory: T:\Sammy Data\Third set results\out_anndata_peptides
Modality: peptides


## Discover Samples

In [ ]:
# Find all samples (auto-detect format)
from utils.io import detect_parquet_format

# Check for single-file format first (parquet files at root level)
single_files = list(PARQUET_DIR.glob("*.parquet"))
single_files = [f for f in single_files if f.stem != "extraction_summary"]

# Check for per-channel format (subdirectories with parquet files)
sample_dirs = [d for d in PARQUET_DIR.iterdir() if d.is_dir()]

if single_files:
    # Single-file format detected
    sample_ids = [f.stem for f in single_files]
    detected_format = "single-file"
    print(f"Detected format: single-file (one parquet per sample)")
    print(f"Found {len(sample_ids)} samples:")
    for sid in sample_ids[:10]:  # Show first 10
        f = PARQUET_DIR / f"{sid}.parquet"
        size_mb = f.stat().st_size / (1024 * 1024)
        # Count columns (channels)
        df_cols = pd.read_parquet(f, columns=None).columns
        n_channels = len([c for c in df_cols if c not in ('x', 'y')])
        print(f"  - {sid}: {n_channels} channels, {size_mb:.1f} MB")
    if len(sample_ids) > 10:
        print(f"  ... and {len(sample_ids) - 10} more")
elif sample_dirs:
    # Per-channel format detected
    sample_ids = [d.name for d in sample_dirs]
    detected_format = "per-channel"
    print(f"Detected format: per-channel (directory per sample)")
    print(f"Found {len(sample_ids)} samples:")
    for sid in sample_ids[:10]:
        n_files = len(list((PARQUET_DIR / sid).glob("*.parquet")))
        print(f"  - {sid}: {n_files} channels")
    if len(sample_ids) > 10:
        print(f"  ... and {len(sample_ids) - 10} more")
else:
    print("No samples found!")
    sample_ids = []

## Process Single Sample (Example)

In [ ]:
def process_sample(sample_id: str, parquet_dir: Path, output_dir: Path, 
                   modality: str, join_type: str = "outer", 
                   fill_value: float = 0.0) -> ad.AnnData:
    """Process a single sample: load data (auto-detecting format), create AnnData."""
    
    logger.info(f"Processing sample: {sample_id}")
    
    # Use the new auto-detecting function
    adata = msi_io.load_msi_to_anndata(
        input_dir=parquet_dir,
        sample_id=sample_id,
        modality=modality,
        fill_value=fill_value,
    )
    
    logger.info(f"  Created AnnData: {adata.n_obs} pixels x {adata.n_vars} channels")
    
    # Save
    output_path = output_dir / f"{sample_id}.h5ad"
    msi_io.save_anndata(adata, output_path)
    
    return adata

In [24]:
# Process first sample as example
if sample_ids:
    example_sample = sample_ids[0]
    print(f"Processing example sample: {example_sample}")
    
    adata = process_sample(
        sample_id=example_sample,
        parquet_dir=PARQUET_DIR,
        output_dir=OUTPUT_DIR,
        modality=MODALITY,
        join_type=JOIN_TYPE,
        fill_value=FILL_VALUE
    )
    
    print(f"\nCreated AnnData:")
    print(f"  Shape: {adata.shape}")
    print(f"  Observations (pixels): {adata.n_obs}")
    print(f"  Variables (channels): {adata.n_vars}")
    print(f"  Spatial range X: [{adata.obsm['spatial'][:, 0].min():.0f}, {adata.obsm['spatial'][:, 0].max():.0f}]")
    print(f"  Spatial range Y: [{adata.obsm['spatial'][:, 1].min():.0f}, {adata.obsm['spatial'][:, 1].max():.0f}]")

INFO:__main__:Processing sample: S001 ovary manual peptides
INFO:utils.io:Found 773 parquet files in T:\Sammy Data\Third set results\out_msi_peptides\S001 ovary manual peptides


Processing example sample: S001 ovary manual peptides


INFO:__main__:  Loaded 773 channels
INFO:utils.io:Merged 773 channels: 270105 pixels x 773 channels
INFO:__main__:  Merged data: 270105 pixels x 773 channels
c:\Users\users_\.conda\envs\maitra_spatial\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
INFO:utils.io:Created AnnData: 270105 pixels x 773 channels
... storing 'sample_id' as categorical
... storing 'modality' as categorical
INFO:utils.io:Saved AnnData to T:\Sammy Data\Third set results\out_anndata_peptides\S001 ovary manual peptides.h5ad



Created AnnData:
  Shape: (270105, 773)
  Observations (pixels): 270105
  Variables (channels): 773
  Spatial range X: [0, 635]
  Spatial range Y: [0, 529]


## Inspect AnnData Structure

In [25]:
if 'adata' in dir():
    print("AnnData structure:")
    print(adata)
    
    print("\n.obs columns:")
    print(adata.obs.head())
    
    print("\n.var (channel info):")
    print(adata.var.head(10))
    
    print("\n.obsm keys:")
    print(list(adata.obsm.keys()))
    
    print("\n.uns metadata:")
    print(adata.uns.get('msi_metadata', {}))

AnnData structure:
AnnData object with n_obs × n_vars = 270105 × 773
    obs: 'x', 'y', 'sample_id'
    var: 'channel_name', 'modality'
    uns: 'msi_metadata'
    obsm: 'spatial'

.obs columns:
   x    y                   sample_id
0  0  177  S001 ovary manual peptides
1  1  175  S001 ovary manual peptides
2  1  176  S001 ovary manual peptides
3  1  177  S001 ovary manual peptides
4  1  178  S001 ovary manual peptides

.var (channel info):
                                  channel_name  modality
1001_3598_m_z___15_ppm  1001_3598_m_z___15_ppm  peptides
1002_517_m_z___15_ppm    1002_517_m_z___15_ppm  peptides
1005_6146_m_z___15_ppm  1005_6146_m_z___15_ppm  peptides
1008_53_m_z___15_ppm      1008_53_m_z___15_ppm  peptides
1010_4986_m_z___15_ppm  1010_4986_m_z___15_ppm  peptides
1010_534_m_z___15_ppm    1010_534_m_z___15_ppm  peptides
1013_4384_m_z___15_ppm  1013_4384_m_z___15_ppm  peptides
1013_5042_m_z___15_ppm  1013_5042_m_z___15_ppm  peptides
1013_5752_m_z___15_ppm  1013_5752_m_z___15

## Batch Process All Samples

In [26]:
# Process all samples
results = []

for sample_id in sample_ids:
    try:
        adata = process_sample(
            sample_id=sample_id,
            parquet_dir=PARQUET_DIR,
            output_dir=OUTPUT_DIR,
            modality=MODALITY,
            join_type=JOIN_TYPE,
            fill_value=FILL_VALUE
        )
        
        results.append({
            'sample_id': sample_id,
            'n_pixels': adata.n_obs,
            'n_channels': adata.n_vars,
            'status': 'success'
        })
        
    except Exception as e:
        logger.error(f"Error processing {sample_id}: {e}")
        results.append({
            'sample_id': sample_id,
            'status': 'error',
            'error': str(e)
        })

# Summary
results_df = pd.DataFrame(results)
print("\nProcessing Summary:")
print(results_df)

INFO:__main__:Processing sample: S001 ovary manual peptides
INFO:utils.io:Found 773 parquet files in T:\Sammy Data\Third set results\out_msi_peptides\S001 ovary manual peptides
INFO:__main__:  Loaded 773 channels
INFO:utils.io:Merged 773 channels: 270105 pixels x 773 channels
INFO:__main__:  Merged data: 270105 pixels x 773 channels
c:\Users\users_\.conda\envs\maitra_spatial\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
INFO:utils.io:Created AnnData: 270105 pixels x 773 channels
... storing 'sample_id' as categorical
... storing 'modality' as categorical
INFO:utils.io:Saved AnnData to T:\Sammy Data\Third set results\out_anndata_peptides\S001 ovary manual peptides.h5ad
INFO:__main__:Processing sample: S004 ovary manual peptides
INFO:utils.io:Found 773 parquet files in T:\Sammy Data\Third set results\out_msi_peptides\S004 ovary manual peptides
INFO:__main__


Processing Summary:
                     sample_id  n_pixels  n_channels   status
0   S001 ovary manual peptides    270105         773  success
1   S004 ovary manual peptides    165017         773  success
2   S005 ovary manual peptides    219899         773  success
3   S006 ovary manual peptides    188957         773  success
4   S019 ovary manual peptides    169205         773  success
5   S020 ovary manual peptides    116189         773  success
6   S022 ovary manual peptides    121646         773  success
7   S023 ovary manual peptides    214975         773  success
8   S024 ovary manual peptides    198956         773  success
9   S025 ovary manual peptides    223183         773  success
10  S026 ovary manual peptides    122290         773  success
11  S030 ovary manual peptides    239853         773  success
12  S032 ovary manual peptides    190864         773  success
13  S033 ovary manual peptides    247345         773  success
14  S034 ovary manual peptides    220055         

## Verify Output Files

In [27]:
# List generated h5ad files
h5ad_files = list(OUTPUT_DIR.glob("*.h5ad"))

print(f"Generated {len(h5ad_files)} AnnData files:")
for f in h5ad_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name}: {size_mb:.1f} MB")

Generated 40 AnnData files:
  - S001 ovary manual peptides.h5ad: 386.1 MB
  - S004 ovary manual peptides.h5ad: 247.4 MB
  - S005 ovary manual peptides.h5ad: 369.9 MB
  - S006 ovary manual peptides.h5ad: 319.5 MB
  - S019 ovary manual peptides.h5ad: 244.8 MB
  - S020 ovary manual peptides.h5ad: 192.1 MB
  - S022 ovary manual peptides.h5ad: 192.9 MB
  - S023 ovary manual peptides.h5ad: 311.9 MB
  - S024 ovary manual peptides.h5ad: 309.1 MB
  - S025 ovary manual peptides.h5ad: 357.8 MB
  - S026 ovary manual peptides.h5ad: 191.4 MB
  - S030 ovary manual peptides.h5ad: 330.7 MB
  - S032 ovary manual peptides.h5ad: 256.8 MB
  - S033 ovary manual peptides.h5ad: 374.5 MB
  - S034 ovary manual peptides.h5ad: 321.2 MB
  - S044 ovary manual peptides.h5ad: 244.4 MB
  - S045 ovary manual peptides.h5ad: 307.4 MB
  - S046 ovary manual peptides.h5ad: 129.3 MB
  - S047 ovary manual peptides.h5ad: 236.6 MB
  - SO11 ovary manual peptides.h5ad: 160.2 MB
  - SO14 ovary manual peptides.h5ad: 425.9 MB
  - SO

In [28]:
# Test loading a saved file
if h5ad_files:
    test_file = h5ad_files[0]
    print(f"Loading {test_file.name} to verify...")
    
    adata_loaded = ad.read_h5ad(test_file)
    print(f"  Shape: {adata_loaded.shape}")
    print(f"  Spatial coordinates present: {'spatial' in adata_loaded.obsm}")
    print("  Verification successful!")

Loading S001 ovary manual peptides.h5ad to verify...
  Shape: (270105, 773)
  Spatial coordinates present: True
  Verification successful!
